# 🎬 VideoRAG 프로토타입 — 환경 설정 (00_setup)
#
# 역할: GPU 확인, Google Drive 마운트, 코드 동기화, 전체 의존성 설치, InternVideo2 클론, import 검증
# 실행 시점: 코랩 세션 시작 시 1회
#
# 변경사항 (v2):
#   - MSR-VTT 전체 영상 사용 (샘플링 X)
#   - 한글 캡션 경로 설정 추가
#   - 한글 형태소 분석기(konlpy) 의존성 추가
#   - 경로 설정을 상단 CONFIG 셀로 통합
#
# Step 1: GPU 확인 & Drive 마운트
# Step 1.5: 경로 CONFIG (영상, 캡션, 인덱스 등 모든 경로를 여기서 관리)
# Step 2: 프로젝트 코드 가져오기
# Step 3: 전체 의존성 설치 (konlpy 포함)
# Step 3.5: InternVideo2 sparse checkout
# Step 4: sys.path 등록 & src 모듈 import 검증

In [5]:
# ── Step 1: GPU 확인 & Google Drive 마운트 ──
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive 마운트 완료")

Mon Mar 30 05:52:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# ── Step 1.5: 경로 CONFIG (모든 경로를 여기서 관리) ──
# ⚠ 새 환경에서 실행 시 이 셀의 경로만 수정하면 됩니다.

import os

# ── 프로젝트 루트 ──
PROJECT_ROOT   = '/content/videorag_prototype'
DRIVE_PROJECT  = '/content/drive/MyDrive/videorag_prototype'

# ── MSR-VTT 데이터 ──
DATA_DIR       = os.path.join(PROJECT_ROOT, 'data', 'msrvtt')
VIDEO_DIR      = os.path.join(DATA_DIR, 'videos')           # 전체 MSR-VTT 영상
KEYFRAME_DIR   = os.path.join(DATA_DIR, 'keyframes')             # 추출된 키프레임

# ── 인덱스 & 출력 ──
INDEX_DIR      = os.path.join(PROJECT_ROOT, 'index')
OUTPUT_DIR     = os.path.join(PROJECT_ROOT, 'output')

# ── Drive 경로 ──
DRIVE_INDEX    = os.path.join(DRIVE_PROJECT, 'index')
DRIVE_VIDEOS   = os.path.join(DRIVE_PROJECT, 'data', 'msrvtt', 'videos')
DRIVE_CAPTIONS_DIR = os.path.join(DRIVE_PROJECT, 'data', 'msrvtt', 'videos', 'msrvtt_captions')

# ── 디렉토리 생성 ──
for d in [DATA_DIR, VIDEO_DIR, KEYFRAME_DIR, INDEX_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 설정 요약 출력 ──
print("═" * 60)
print("📋 VideoRAG 경로 CONFIG")
print("═" * 60)
print(f"  프로젝트:       {PROJECT_ROOT}")
print(f"  영상 (전체) :  {VIDEO_DIR}")
print(f"  키프레임:       {KEYFRAME_DIR}")
print(f"  인덱스:         {INDEX_DIR}")
print(f"  출력:           {OUTPUT_DIR}")
print(f"  Drive 캡션:     {DRIVE_CAPTIONS_DIR}")
print("═" * 60)

# 영상 파일 수 확인
if os.path.exists(VIDEO_DIR):
    mp4_count = len([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])
    print(f"✓ 영상 파일: {mp4_count}개")
else:
    print(f"⚠ 영상 디렉토리 없음: {VIDEO_DIR}")


In [7]:
# ── Step 2: 프로젝트 코드 가져오기 ──
import os, shutil
from google.colab import userdata

# [Public repo] token 불필요 — public clone

LOCAL_PROJECT = '/content/videorag_prototype'

# (1) 프로젝트 폴더 준비 (Drive → 로컬 복사 or git clone)
if not os.path.exists(LOCAL_PROJECT):
    if os.path.exists(DRIVE_PROJECT):
        shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
        print('✓ Drive → 로컬 복사 완료')
    else:
        os.system(f"git clone https://github.com/LimPark996/VideoRAG-Public.git {LOCAL_PROJECT}")
        print('✓ git clone 완료')
else:
    print(f'✓ 프로젝트 이미 존재: {LOCAL_PROJECT}')

# (2) GitHub에서 최신 src 동기화 (항상 최신 코드로 덮어쓰기)
os.system("rm -rf /tmp/VideoRAG-Prototype")
os.system("git clone https://github.com/LimPark996/VideoRAG-Public.git /tmp/VideoRAG-Prototype")
os.system(f"rm -rf {LOCAL_PROJECT}/src")
os.system(f"cp -r /tmp/VideoRAG-Prototype/src {LOCAL_PROJECT}/src")
print('✓ GitHub에서 최신 src 동기화 완료')

# (3) HF_TOKEN 설정 (InternVideo2 모델 다운로드에 필수)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print('✓ HF_TOKEN 설정 완료')

✓ 프로젝트 이미 존재: /content/videorag_prototype
✓ GitHub에서 최신 src 동기화 완료
✓ HF_TOKEN 설정 완료


In [ ]:
# ── Step 3: 전체 의존성 설치 ──

# InternVideo2 의존성 (open_clip 필수)
!pip install open_clip_torch

# Core ML
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 2>/dev/null

# Search, Reranking, Video, C2PA, UI, Eval, Utilities
!pip install -q transformers timm einops rank_bm25 faiss-cpu \
    moviepy opencv-python cryptography gradio scikit-learn scipy \
    tqdm matplotlib pandas numpy easydict 2>/dev/null

# ColBERT v2 — Phase 3 리랭킹에 필수
!pip install -q ragatouille colbert-ai 2>/dev/null

# TransNetV2 (Shot 탐지)
!git clone -q https://github.com/soCzech/TransNetV2 /content/TransNetV2 2>/dev/null
!pip install -q -e /content/TransNetV2/ 2>/dev/null

# TransNetV2 가중치 다운로드 (Git LFS 포인터만 받아지는 문제 해결)
import os
TRANSNET_WEIGHTS = '/content/TransNetV2/inference/transnetv2-weights'
os.makedirs(TRANSNET_WEIGHTS, exist_ok=True)
WEIGHT_URL = 'https://github.com/soCzech/TransNetV2/raw/master/inference/transnetv2-weights'
for fname in ['saved_model.pb', 'variables/variables.index', 'variables/variables.data-00000-of-00001']:
    fpath = os.path.join(TRANSNET_WEIGHTS, fname)
    if not os.path.exists(fpath) or os.path.getsize(fpath) < 1000:
        os.makedirs(os.path.dirname(fpath), exist_ok=True)
        !wget -q -O "{fpath}" "{WEIGHT_URL}/{fname}"
        print(f'  ✓ {fname} ({os.path.getsize(fpath)/1024/1024:.1f} MB)')
print('✓ TransNetV2 가중치 준비 완료')

# InternVideo2 (HuggingFace 모델 다운로드용)
!pip install -q huggingface_hub>=0.19.0

# ── v2 추가 의존성 ──
!pip install -q langdetect>=1.0.9
!pip install -q requests>=2.28.0
!pip install -q openai>=1.0.0

print("\n✓ 전체 의존성 설치 완료")

In [ ]:
# ── Step 3.5: InternVideo2 설치 (런타임 초기화 시 1회) ──
import shutil, os, sys, importlib

# 삭제 전에 안전한 디렉토리로 이동
os.chdir('/content')

# 기존 디렉토리 삭제 (클린 재설치)
if os.path.exists('/content/InternVideo'):
    shutil.rmtree('/content/InternVideo')
    print('✓ 기존 /content/InternVideo 삭제 완료')

# sparse checkout 클론 (전체 레포 대신 multi_modality만)
!git clone --no-checkout --depth=1 https://github.com/OpenGVLab/InternVideo.git /content/InternVideo
%cd /content/InternVideo
!git sparse-checkout init --cone
!git sparse-checkout set InternVideo2/multi_modality
!git checkout main

# cwd 복원
os.chdir('/content')

# sys.path 등록 + import 캐시 무효화
INTERNVIDEO_PATH = '/content/InternVideo/InternVideo2/multi_modality'
if INTERNVIDEO_PATH not in sys.path:
    sys.path.insert(0, INTERNVIDEO_PATH)
importlib.invalidate_caches()
print(f'✓ sys.path에 {INTERNVIDEO_PATH} 추가')

# 확인
print(os.listdir(INTERNVIDEO_PATH))

In [ ]:
# ── Step 4: sys.path 등록 & src 모듈 import 검증 ──
import sys, os

LOCAL_PROJECT = '/content/videorag_prototype'
sys.path.insert(0, LOCAL_PROJECT)
sys.path.insert(0, '/content/TransNetV2/inference')
# InternVideo2는 Step 3.5에서 이미 등록됨

os.chdir('/content')  # cwd 복원

import torch
import numpy as np
import faiss
import cv2

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# src 모듈 import 테스트
from src.data_models import ClipMeta, ClipResult, VideoRAGResult
from src.phase0_indexing.shot_detector import ShotDetector, SceneInfo
from src.phase0_indexing.embedder import VideoEmbedder
from src.phase0_indexing.vector_store import FAISSVectorStore
from src.phase0_indexing.indexer import VideoIndexer
from src.phase12_search.bm25_retriever import BM25Retriever
from src.phase12_search.dense_retriever import DenseRetriever
from src.phase12_search.hybrid_fusion import HybridFusion
from src.phase3_reranking.reranker import ColBERTReranker, PLAIDEngine
from src.phase4_assembly.assembler import VideoAssembler
from src.phase4_assembly.visual_scorer import VisualScorer
from src.phase4_assembly.transition_selector import TransitionSelector
from src.phase4_assembly.colour_normalizer import ColourNormalizer, LUT3D
from src.phase5_c2pa.c2pa_tagger import C2PATagger
from src.hallucination.hallucination_detector import HallucinationDetector, PRISMDetector, WCSDetector
from src.input import QueryPreprocessor
from src.output import TimelineExporter
from src.pipeline import VideoRAGPipeline

print("\n✓ 전체 src 모듈 import 성공!")
print("  Phase 0: ShotDetector, VideoEmbedder, FAISSVectorStore, VideoIndexer")
print("  Phase 1-2: BM25Retriever, DenseRetriever, HybridFusion(WRRF)")
print("  Phase 3: ColBERTReranker + PLAIDEngine")
print("  Phase 4: VideoAssembler, VisualScorer(DINOv2), TransitionSelector, ColourNormalizer(3D LUT)")
print("  Phase 5: C2PATagger(ES256)")
print("  입출력: QueryPreprocessor, TimelineExporter")
print("  환각탐지: HallucinationDetector (PRISM + WCS 이중 검증)")
print("  Pipeline: VideoRAGPipeline")

# QueryPreprocessor 테스트
try:
    qp = QueryPreprocessor()
    print("\n✓ QueryPreprocessor 인스턴스화 성공")
except Exception as e:
    print(f"\n⚠ QueryPreprocessor 에러: {e}")